In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print(f"Çalışma Ortamı: {'GPU (T4)' if device == 0 else 'CPU'}")

# --- 1. MODEL BORU HATLARININ (PIPELINES) KURULUMU ---
# A) Metin Sınıflandırma (Çok dilli XLM-RoBERTa tabanlı Zero-Shot Sınıflandırıcı)
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=device
)

# B) Varlık Tanıma / NER (Türkçe BERT NER modeli)
ner_tagger = pipeline(
    "ner",
    model="savasy/bert-base-turkish-ner-cased",
    aggregation_strategy="simple",
    device=device
)

# C) Özetleme (Transformers v5 uyumlu text-generation pipeline ile Qwen2.5-1.5B-Instruct)
ozet_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device=device
)

# --- 2. ÖRNEK SAHA / OPERASYON RAPORU ---
saha_raporu = (
    "Bugün sabah saat 08:45'te Çorlu Uçuş Eğitim ve Test Merkezi'nden havalanan "
    "Bayraktar KIZILELMA prototipi, 12.000 feet irtifada seyir halindeyken Yer Kontrol "
    "İstasyonu ile C-Bant veri linki bağlantısında 8 saniyelik anlık sinyal zayıflaması "
    "yaşamıştır. Aviyonik sistemler otomatik yedek frekansa geçiş yapmış olup uçuş "
    "güvenliğini tehlikeye atacak herhangi bir durum oluşmamış ve test görevi "
    "planlandığı şekilde tamamlanmıştır."
)

# --- 3. UÇTAN UCA PIPELINE ÇALIŞTIRMA ---
def uctan_uca_nlp_isle(metin):
    print("=" * 65)
    print("HAM SAHA RAPORU:\n", metin)
    print("=" * 65)

    # ADIM 1: Metin Sınıflandırma (Arıza / Talep Kategorisi)
    aday_etiketler = [
        "Haberleşme ve Datalink",
        "Motor ve İtki Sistemi",
        "Batarya ve Güç Yönetimi",
        "İniş Takımları ve Frenleme"
    ]
    sinif_sonuc = classifier(metin, candidate_labels=aday_etiketler)
    en_iyi_etiket = sinif_sonuc["labels"][0]
    guven_skoru = sinif_sonuc["scores"][0]

    print(f"\n[1] METİN SINIFLANDIRMA SONUCU:")
    print(f"    -> Kategori    : {en_iyi_etiket} (Güven: %{guven_skoru*100:.1f})")

    # ADIM 2: Varlık Tanıma (NER - Kurum / Yer / Platform Çıkarımı)
    varliklar = ner_tagger(metin)
    print(f"\n[2] VARLIK TANIMA (NER) SONUÇLARI:")
    if varliklar:
        for v in varliklar:
            print(f"    -> {v['word']:<35} | Tür: {v['entity_group']} (Skor: {v['score']:.2f})")
    else:
        print("    -> Belirgin varlık bulunamadı.")

    # ADIM 3: Özetleme (Komuta Kontrol Özeti)
    mesajlar = [
        {"role": "system", "content": "Sen bir askeri havacılık rapor özetleyicisisin. Verilen raporu kritik teknik detayları koruyarak tek cümlede özetle."},
        {"role": "user", "content": metin}
    ]
    ozet_cikti = ozet_pipeline(mesajlar, max_new_tokens=120, do_sample=False)
    ozet_metni = ozet_cikti[0]["generated_text"][-1]["content"]

    print(f"\n[3] ÖZETLEME SONUCU:")
    print(f"    -> {ozet_metni}")
    print("=" * 65)

uctan_uca_nlp_isle(saha_raporu)

Çalışma Ortamı: GPU (T4)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: savasy/bert-base-turkish-ner-cased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

HAM SAHA RAPORU:
 Bugün sabah saat 08:45'te Çorlu Uçuş Eğitim ve Test Merkezi'nden havalanan Bayraktar KIZILELMA prototipi, 12.000 feet irtifada seyir halindeyken Yer Kontrol İstasyonu ile C-Bant veri linki bağlantısında 8 saniyelik anlık sinyal zayıflaması yaşamıştır. Aviyonik sistemler otomatik yedek frekansa geçiş yapmış olup uçuş güvenliğini tehlikeye atacak herhangi bir durum oluşmamış ve test görevi planlandığı şekilde tamamlanmıştır.

[1] METİN SINIFLANDIRMA SONUCU:
    -> Kategori    : Haberleşme ve Datalink (Güven: %98.4)

[2] VARLIK TANIMA (NER) SONUÇLARI:
    -> Çorlu                               | Tür: LOC (Skor: 0.97)
    -> Bayraktar KIZILELMA                 | Tür: ORG (Skor: 0.69)
    -> Yer Kontrol İstasyonu               | Tür: ORG (Skor: 0.89)
    -> C - Bant                            | Tür: ORG (Skor: 0.68)

[3] ÖZETLEME SONUCU:
    -> Bayraktar KIZILELMA prototipi, saat 08:45'da Çorlu Uçuş Eğitim ve Test Merkezi'nde havalanırken Yer Kontrol İstasyonu ile C-Bant v